# Gaussian Novelty

Fit a multivariate normal distribution to clean training data and flag new points with a large Mahalanobis distance. If the data really is normal, the chi-square distribution gives a threshold with an exact false-alarm rate.

In [ ]:
using Distributions, LinearAlgebra, Plots, Statistics, Random

Random.seed!(2);

## Correlated Training Data

Two strongly correlated features. One test point has ordinary values in each feature but breaks the correlation.

In [ ]:
Sigma = [1.0 0.9; 0.9 1.0]
X_train = rand(MvNormal([0.0, 0.0], Sigma), 500)'
X_test  = vcat(rand(MvNormal([0.0, 0.0], Sigma), 30)', [1.5 -1.5; 3.5 3.5; 0.0 3.0])
is_true = vcat(falses(30), trues(3))

## Mahalanobis Distance

The squared distance from the training mean, in units of the training covariance.

In [ ]:
mu     = vec(mean(X_train, dims = 1))
SigInv = inv(cov(X_train))
mahal(x) = dot(x .- mu, SigInv * (x .- mu))
scores = [mahal(x) for x in eachrow(X_test)]

alpha = 0.01
threshold = quantile(Chisq(size(X_train, 2)), 1 - alpha)
is_novel = scores .> threshold
println("Threshold: ", round(threshold, digits = 2))
println("Novel test points: ", findall(is_novel))

## Mahalanobis Versus Euclidean

Point (1.5, -1.5) is close to the mean in plain distance but lies across the correlation, so its Mahalanobis distance is large.

In [ ]:
euclid = [sum(abs2, x .- mu) for x in eachrow(X_test)]
for i in findall(is_true)
    println("Point ", X_test[i, :], "  Euclidean^2: ", round(euclid[i], digits = 2), "  Mahalanobis^2: ", round(scores[i], digits = 2))
end

In [ ]:
xs = range(-4, 4; length = 100)
contour(xs, xs, (a, b) -> mahal([a, b]); levels = [threshold], color = :black, linewidth = 2, colorbar = false)
scatter!(X_train[:, 1], X_train[:, 2]; color = :lightgray, markerstrokewidth = 0, label = "Training")
scatter!(X_test[is_novel, 1], X_test[is_novel, 2]; color = :red, marker = :star5, label = "Novel")

## Is the Threshold Calibrated?

On normal training data about alpha of the points should exceed the threshold. A Q-Q plot of the distances against the chi-square distribution checks the normality assumption.

In [ ]:
train_scores = [mahal(x) for x in eachrow(X_train)]
println("Training points above threshold: ", round(100 * mean(train_scores .> threshold), digits = 1), "% (alpha = 1%)")

n = length(train_scores)
theory = quantile.(Chisq(2), ((1:n) .- 0.5) ./ n)
scatter(theory, sort(train_scores); markerstrokewidth = 0, label = "", xlabel = "Chi-square quantile", ylabel = "Mahalanobis^2")
plot!(theory, theory; color = :red, label = "y = x")

## Summary
- The Mahalanobis distance measures distance in units of the training covariance.
- It catches points that break correlations, which per-feature rules miss.
- The chi-square quantile gives a threshold with false-alarm rate alpha, if the data is normal.
- Check the calibration on training data before relying on it.